SECTION A: CONCEPT APPLICATION

Question 1
Scenario: You are a Data Analyst at a food delivery startup. You have loaded a dataset of 50,000 order records into a Pandas DataFrame. The delivery_time_mins column has 14% missing values, and the customer_rating column contains some entries stored as strings (e.g., '4.5') instead of float64.

Answer:

Handling missing values in delivery_time_mins (14% missing):
I would choose median imputation instead of dropping rows. Dropping 14% of a 50,000-row dataset means throwing away around 7,000 valid records, which is a massive waste of data. Also, delivery times tend to be right-skewed because of traffic spikes or long-distance orders. The median is much better than the mean here because it isn't dragged down or thrown off by extreme outlier values.

Fixing customer_rating column type:
Before doing any analysis, you need to cast those string ratings into numeric float values so math and aggregation functions work properly.

In [ ]:
df['customer_rating'] = pd.to_numeric(df['customer_rating'], errors='coerce')

Question 2
Scenario: You are building a weekly performance report for a food delivery platform. You have a NumPy array of daily order counts across 30 days. Your manager wants to flag 'surge days'—days where the order count exceeds the mean by more than 1.5 standard deviations—and compute the total revenue only for those surge days using a matching revenue array.

Answer:
NumPy handles this efficiently using vectorized operations and boolean masks. Functions like np.mean() and np.std() calculate stats across the whole array instantly in compiled C code under the hood. When you apply a condition like daily_orders > threshold, NumPy generates a boolean mask (an array of True/False values matching the condition).
This is way faster than writing a standard Python for loop to check each day one-by-one because it bypasses Python's slow interpreter loop overhead and leverages low-level hardware optimizations.

Question 3
Scenario: You are conducting EDA for a food delivery product team. Your dataset contains columns: restaurant_category, average_order_value, customer_rating, and delivery_time_mins. The team wants to know (a) whether delivery time is correlated with customer rating, and (b) how average order value is distributed across restaurant categories.

Answer:

(a) Delivery time vs. customer rating: This is a bivariate analysis comparing two continuous numerical variables. The best chart to use is a Seaborn scatter plot with a regression line (sns.scatterplot or sns.regplot). A line plot would be wrong because it implies a continuous time-series sequence, whereas a scatter plot clearly displays correlation, spread, and clustering between the two metrics.

(b) Average order value distribution across restaurant categories: This is a categorical vs. numerical comparison. The best chart is a Seaborn boxplot (sns.boxplot). A boxplot is ideal because it instantly shows the median, interquartile range (IQR), and outliers of order values for every category side-by-side, giving a complete picture of the distributions.

uestion 4
Scenario: You are managing a food delivery database with three tables: orders, restaurants, and customers. A business team asks for a report showing every restaurant's name and total revenue, including restaurants that have not yet received any orders.

Answer:
You must use a LEFT JOIN (with restaurants as the left table and orders as the right table).
If you use an INNER JOIN, it only returns restaurants that already have matching order records, which would completely drop any new or zero-order restaurants from the report. A LEFT JOIN keeps all records from the restaurants table intact and attaches order data where it exists, putting NULL where there are no matches. This ensures zero-order restaurants still show up on the report.

Question 5
Scenario: You are working on a food delivery analytics dashboard. Management wants to rank all delivery agents within each city by their number of completed deliveries for the current month, and also display how each agent's count compares to the agent ranked directly above them in that city.

Answer:
You would use the RANK() window function for the ranking and the LAG() window function to pull the value from the previous row.
Using window functions with PARTITION BY city is much better than a GROUP BY or self-join approach. A GROUP BY would collapse the rows and make it impossible to see individual agent details alongside city-wide stats, while self-joins and correlated subqueries become messy, unreadable, and slow. Window functions calculate everything cleanly in a single pass while keeping row-level details intact.

Question 6Scenario: You are optimising a SQL report for a food delivery operations team. The report first identifies the top 5 restaurant categories by total order count, and then retrieves the full order details only for restaurants belonging to those top categories. A colleague has written this as a deeply nested subquery inside the WHERE clause.Answer:
Using a CTE (WITH clause) is much better than a nested subquery for this.
CTEs break complex queries into readable, named blocks that flow naturally from top to bottom, making it way easier to debug if something goes wrong. Plus, you can reference a CTE multiple times if needed. Deeply nested subqueries become unreadable "spaghetti code" very quickly. You would generally only use a nested subquery for quick, simple inline checks where writing a standalone CTE would be unnecessary overhead.

SECTION B: PRACTICAL CODING TASKS

Task 1: Daily Order Analysis with NumPy
Build a NumPy-based program that analyses 30 days of order count and revenue data for a
food delivery platform.
Create two NumPy arrays of 30 elements each using np.random.seed(42) for reproducibility —
one for daily order counts (integers between 100–800) and one for daily revenue (floats
between 5000–50000).
Calculate and print: total orders, total revenue, mean and standard deviation of daily orders,
and the day number (index) with the highest single-day revenue.
Use boolean indexing to extract all days where order count exceeded mean + 1 standard
deviation. Print the count of such peak days and their total combined revenue.
Reshape the 30-element order array into a 5×6 matrix and use np.sum(axis=1) to print the
total orders for each of the 5 weekly blocks.

In [ ]:
import numpy as np

# Step 1: Set random seed for reproducibility and generate data
np.random.seed(42)

# Generate 30 days of order counts (integers between 100 and 800)
daily_orders = np.random.randint(100, 801, size=30)

# Generate 30 days of daily revenue (floats between 5,000 and 50,000)
daily_revenue = np.random.uniform(5000.0, 50000.0, size=30)

# Step 2: Calculate basic metrics
total_orders = np.sum(daily_orders)
total_revenue = np.sum(daily_revenue)
mean_orders = np.mean(daily_orders)
std_orders = np.std(daily_orders)

max_rev_day = np.argmax(daily_revenue) + 1
max_rev_amount = np.max(daily_revenue)

print("--- [Task 1] Basic Statistics ---")
print(f"Total Orders (30 days): {total_orders}")
print(f"Total Revenue (30 days): ₹{total_revenue:,.2f}")
print(f"Mean Daily Orders: {mean_orders:.2f} | Std Dev: {std_orders:.2f}")
print(f"Highest Revenue Day: Day {max_rev_day} (₹{max_rev_amount:,.2f})")

# Step 3: Boolean indexing for 'surge days' (orders > mean + 1.5 * std)
surge_threshold = mean_orders + (1.5 * std_orders)
surge_mask = daily_orders > surge_threshold
surge_revenue = np.sum(daily_revenue[surge_mask])

print("\n--- [Task 1] Surge Days Analysis ---")
print(f"Surge Threshold: {surge_threshold:.2f} orders")
print(f"Total Surge Days: {np.sum(surge_mask)}")
print(f"Combined Revenue on Surge Days: ₹{surge_revenue:,.2f}")

# Step 4: Reshape into a 5x6 matrix and calculate weekly block totals
weekly_matrix = daily_orders.reshape(5, 6)
weekly_totals = np.sum(weekly_matrix, axis=1)

print("\n--- [Task 1] Weekly Totals (5x6 Matrix) ---")
for week_idx, w_total in enumerate(weekly_totals, start=1):
    print(f"Week {week_idx} Total Orders: {w_total}")

Task 2: Pandas Data Cleaning Pipeline

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Create a simulated DataFrame with missing values and duplicates
raw_data = {
    'order_id': list(range(101, 121)),
    'customer_name': [
        'Aarav', 'Priya', 'Rohan', 'Sneha', 'Vikram',
        'Ananya', 'Kabir', 'Meera', 'Arjun', 'Neha',
        'Karan', 'Pooja', 'Rahul', 'Simran', 'Amit',
        'Divya', 'Kunal', 'Nisha', 'Dev', 'Ritu'
    ],
    'restaurant_name': [
        'Burger King', 'Pizza Hut', 'McDonalds', 'Dominoes', 'Subway',
        'KFC', 'Starbucks', 'Taco Bell', 'Haldirams', 'Pizza Hut',
        'Burger King', 'Pizza Hut', 'McDonalds', 'Dominoes', 'Subway',
        'KFC', 'Starbucks', 'Taco Bell', 'Haldirams', 'Pizza Hut'
    ],
    'category': [
        'Fast Food', 'Italian', 'Fast Food', 'Italian', 'Healthy',
        'Fast Food', 'Cafe', 'Mexican', 'Indian', 'Italian',
        'Fast Food', None, 'Fast Food', 'Italian', 'Healthy',
        'Fast Food', 'Cafe', None, 'Indian', 'Italian'
    ],
    'delivery_time_mins': [
        25.5, 45.0, 30.0, None, 15.0,
        40.0, 20.0, 35.0, 50.0, 22.5,
        28.0, 42.0, None, 60.0, 18.0,
        38.0, 22.0, 33.0, 120.0, 24.0
    ],
    'order_value': [
        350.0, 850.0, 400.0, 600.0, 250.0,
        900.0, 200.0, 450.0, 1100.0, 750.0,
        320.0, 800.0, 410.0, 620.0, 270.0,
        880.0, 210.0, 460.0, 3500.0, 730.0
    ],
    'rating': [
        4.5, 3.8, 4.2, None, 4.9,
        3.5, 4.8, 4.0, 4.1, 4.6,
        4.3, 3.9, 4.2, 4.0, 4.9,
        None, 4.7, 4.0, 4.1, 4.5
    ]
}

df_orders = pd.DataFrame(raw_data)

# Introduce 3 duplicate rows intentionally
df_orders = pd.concat([df_orders, df_orders.iloc[0:3]], ignore_index=True)

print(f"Initial Shape (with duplicates): {df_orders.shape}")

# Step 2: Handle missing values
df_orders['delivery_time_mins'] = df_orders['delivery_time_mins'].fillna(df_orders['delivery_time_mins'].median())
df_orders['order_value'] = df_orders['order_value'].fillna(df_orders['order_value'].median())
df_orders['rating'] = df_orders['rating'].fillna(round(df_orders['rating'].mean(), 1))
df_orders['category'] = df_orders['category'].fillna(df_orders['category'].mode()[0])

# Step 3: Remove duplicate rows
df_orders = df_orders.drop_duplicates()
print(f"Shape After Deduplication: {df_orders.shape}")

# Step 4: IQR Outlier Capping on 'order_value'
Q1 = df_orders['order_value'].quantile(0.25)
Q3 = df_orders['order_value'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

df_orders['order_value'] = df_orders['order_value'].clip(upper=upper_fence)
print(f"Outlier Capping Complete. Upper Fence: ₹{upper_fence:.2f}")
print(f"Max order value after capping: ₹{df_orders['order_value'].max():.2f}")

Task 3: Multi-Panel Visualization DashboardThis script generates a clean $2 \times 2$ subplot grid combining Matplotlib and Seaborn to visualize order counts, delivery time spreads, average order values by restaurant, and a feature correlation matrix heatmap, saving the output as an image.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual theme
sns.set_theme(style="whitegrid")

# Create a 2x2 subplot grid
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Food Delivery Platform - Exploratory Analytics Dashboard', fontsize=16, fontweight='bold')

# Plot 1: Order count by category
sns.countplot(data=df_orders, x='category', ax=axes[0, 0], palette='Set2', hue='category', legend=False)
axes[0, 0].set_title('Order Count by Restaurant Category')
axes[0, 0].set_xlabel('Category')
axes[0, 0].set_ylabel('Total Orders')

# Plot 2: Delivery time distribution by category (Boxplot)
sns.boxplot(data=df_orders, x='category', y='delivery_time_mins', ax=axes[0, 1], palette='Set3', hue='category', legend=False)
axes[0, 1].set_title('Delivery Time Distribution by Category')
axes[0, 1].set_xlabel('Category')
axes[0, 1].set_ylabel('Delivery Time (mins)')

# Plot 3: Average order value per restaurant
avg_val_by_rest = df_orders.groupby('restaurant_name')['order_value'].mean().sort_values()
axes[1, 0].barh(avg_val_by_rest.index, avg_val_by_rest.values, color='cornflowerblue')
axes[1, 0].set_title('Average Order Value per Restaurant')
axes[1, 0].set_xlabel('Average Order Value (₹)')
axes[1, 0].set_ylabel('Restaurant Name')

# Plot 4: Correlation matrix heatmap
numeric_cols = df_orders.select_dtypes(include=[np.number])
corr_matrix = numeric_cols.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 1], linewidths=0.5)
axes[1, 1].set_title('Correlation Matrix of Numeric Features')

plt.tight_layout()
dashboard_filename = 'food_delivery_dashboard.png'
plt.savefig(dashboard_filename, dpi=150, bbox_inches='tight')
plt.show()

print(f"Dashboard successfully generated and saved as '{dashboard_filename}'")

Task 4: SQLite Database & Aggregation Report
This script creates a local SQLite database, sets up related restaurants and orders tables, populates them with relational sample data, runs an aggregation query using a LEFT JOIN and GROUP BY, and exports the final performance report to a CSV file.

In [ ]:
import sqlite3

# Connect to SQLite database file
conn = sqlite3.connect('food_delivery.db')
cursor = conn.cursor()

# Create tables
cursor.execute('DROP TABLE IF EXISTS orders;')
cursor.execute('DROP TABLE IF EXISTS restaurants;')

cursor.execute("""
CREATE TABLE restaurants (
    restaurant_id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    city TEXT
);
""")

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    restaurant_id INTEGER,
    order_value REAL,
    delivery_time_mins REAL,
    rating REAL,
    FOREIGN KEY(restaurant_id) REFERENCES restaurants(restaurant_id)
);
""")

# Insert sample data
restaurants_data = [
    (1, 'Burger King', 'Fast Food', 'Mumbai'),
    (2, 'Pizza Hut', 'Italian', 'Delhi'),
    (3, 'McDonalds', 'Fast Food', 'Bangalore'),
    (4, 'Dominoes', 'Italian', 'Mumbai'),
    (5, 'Subway', 'Healthy', 'Pune'),
    (6, 'KFC', 'Fast Food', 'Delhi'),
    (7, 'Starbucks', 'Cafe', 'Bangalore'),
    (8, 'Haldirams', 'Indian', 'Pune')
]
cursor.executemany('INSERT INTO restaurants VALUES (?, ?, ?, ?);', restaurants_data)

np.random.seed(100)
orders_data = []
for i in range(1, 26):
    r_id = np.random.randint(1, 9)
    val = round(np.random.uniform(200, 1500), 2)
    d_time = round(np.random.uniform(15, 55), 2)
    rate = round(np.random.uniform(3.0, 5.0), 1)
    orders_data.append((i, r_id, val, d_time, rate))

cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?);', orders_data)
conn.commit()

# SQL Query using LEFT JOIN and GROUP BY
query = """
SELECT
    r.name AS restaurant_name,
    r.category,
    r.city,
    COUNT(o.order_id) AS total_orders,
    SUM(o.order_value) AS total_revenue,
    ROUND(AVG(o.rating), 2) AS avg_rating
FROM restaurants r
LEFT JOIN orders o ON r.restaurant_id = o.restaurant_id
GROUP BY r.restaurant_id, r.name, r.category, r.city;
"""

df_report = pd.read_sql_query(query, conn)
conn.close()

# Add rank by revenue
df_report['revenue_rank'] = df_report['total_revenue'].rank(ascending=False, method='min').astype(int)
df_report = df_report.sort_values('revenue_rank', ascending=True)

print("--- Restaurant Performance Report ---")
print(df_report.to_string(index=False))

# Export to CSV
csv_filename = 'restaurant_performance_report.csv'
df_report.to_csv(csv_filename, index=False)
print(f"\nReport successfully exported to '{csv_filename}'")

SECTION C: MINI CAPSTONE & MULTI-SHEET REPORTING

Task 1: Complete Database Initialization & Setup
This script sets up the SQLite database (food_delivery.db), builds the relational schema (restaurants and orders), and seeds it with clean, realistic operational data so everything is ready for analysis.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Step 1: Connect to SQLite database (creates file locally in Colab)
conn = sqlite3.connect('food_delivery.db')
cursor = conn.cursor()

# Step 2: Drop existing tables for clean re-execution
cursor.execute('DROP TABLE IF EXISTS orders;')
cursor.execute('DROP TABLE IF EXISTS restaurants;')

# Step 3: Create relational tables
cursor.execute("""
CREATE TABLE restaurants (
    restaurant_id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    city TEXT
);
""")

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    restaurant_id INTEGER,
    order_value REAL,
    delivery_time_mins REAL,
    rating REAL,
    FOREIGN KEY(restaurant_id) REFERENCES restaurants(restaurant_id)
);
""")

# Step 4: Insert sample restaurant records
restaurants_data = [
    (1, 'Burger King', 'Fast Food', 'Mumbai'),
    (2, 'Pizza Hut', 'Italian', 'Delhi'),
    (3, 'McDonalds', 'Fast Food', 'Bangalore'),
    (4, 'Dominoes', 'Italian', 'Mumbai'),
    (5, 'Subway', 'Healthy', 'Pune'),
    (6, 'KFC', 'Fast Food', 'Delhi'),
    (7, 'Starbucks', 'Cafe', 'Bangalore'),
    (8, 'Haldirams', 'Indian', 'Pune')
]
cursor.executemany("INSERT INTO restaurants VALUES (?, ?, ?, ?);", restaurants_data)

# Step 5: Insert simulated order records with reproducibility
np.random.seed(101)
orders_data = []
for i in range(1, 31): # 30 complete orders
    r_id = np.random.randint(1, 9)
    val = round(np.random.uniform(250, 1400), 2)
    d_time = round(np.random.uniform(18, 50), 2)
    rate = round(np.random.uniform(3.2, 5.0), 1)
    orders_data.append((i, r_id, val, d_time, rate))

cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?);", orders_data)
conn.commit()
conn.close()

print("--- [Task 1] Database Initialized and Populated Successfully ---")

Task 2: Executing Advanced SQL Analytics & KPIs
This script runs a comprehensive SQL query joining the restaurants and orders tables, calculates key business performance metrics (total orders, total revenue, average ratings), and computes revenue rankings.

In [ ]:
# Connect to database and run analytics query
conn = sqlite3.connect('food_delivery.db')

query = """
SELECT
    r.name AS restaurant_name,
    r.category,
    r.city,
    COUNT(o.order_id) AS total_orders,
    SUM(o.order_value) AS total_revenue,
    ROUND(AVG(o.delivery_time_mins), 2) AS avg_delivery_time,
    ROUND(AVG(o.rating), 2) AS avg_rating
FROM restaurants r
LEFT JOIN orders o ON r.restaurant_id = o.restaurant_id
GROUP BY r.restaurant_id, r.name, r.category, r.city
ORDER BY total_revenue DESC;
"""

df_analytics = pd.read_sql_query(query, conn)
conn.close()

# Add a revenue ranking column using Pandas
df_analytics['revenue_rank'] = df_analytics['total_revenue'].rank(ascending=False, method='min').astype(int)
df_analytics = df_analytics.sort_values('revenue_rank')

print("--- [Task 2] Restaurant Performance Leaderboard ---")
print(df_analytics.to_string(index=False))

Task 3: Multi-Sheet Excel Export (Capstone Deliverable)
This script extracts raw and aggregated tables from the SQLite database and uses pandas.ExcelWriter with the openpyxl engine to export a professional multi-sheet Excel report (food_delivery_report.xlsx).

In [ ]:
import openpyxl

# Connect to database to fetch multiple raw datasets
conn = sqlite3.connect('food_delivery.db')

df_orders = pd.read_sql_query("SELECT * FROM orders;", conn)
df_restaurants = pd.read_sql_query("SELECT * FROM restaurants;", conn)

# Re-run performance query for the summary sheet
df_performance = pd.read_sql_query("""
SELECT
    r.name AS restaurant_name,
    r.category,
    r.city,
    COUNT(o.order_id) AS total_orders,
    SUM(o.order_value) AS total_revenue,
    ROUND(AVG(o.rating), 2) AS avg_rating
FROM restaurants r
LEFT JOIN orders o ON r.restaurant_id = o.restaurant_id
GROUP BY r.restaurant_id, r.name, r.category, r.city
ORDER BY total_revenue DESC;
""", conn)

conn.close()

# Export all dataframes into separate sheets of a single Excel file
excel_filename = 'food_delivery_report.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_performance.to_excel(writer, sheet_name='Performance_Summary', index=False)
    df_orders.to_excel(writer, sheet_name='Raw_Orders', index=False)
    df_restaurants.to_excel(writer, sheet_name='Restaurants_Directory', index=False)

print(f"--- [Task 3] Multi-Sheet Excel Report Generated Successfully ---")
print(f"File saved as: '{excel_filename}'")
print("Sheets included: ['Performance_Summary', 'Raw_Orders', 'Restaurants_Directory']")

Task 1: Identifying and Fixing Hidden Bugs in AI-Generated Code
Scenario: An AI tool generated a Python script to calculate monthly delivery performance metrics, but the script is throwing errors or producing incorrect silent results. Below are the two common bugs found in the code and how to fix them.

In [ ]:
# Buggy Code
success_rate = (successful_orders / total_orders) * 100

If successful_orders and total_orders are stored as integers (int), Python performs integer division, returning 0 instead of a float percentage (e.g., 12 / 50 * 100 becomes 0 * 100 = 0).

The Human Fix:
Force float division by multiplying by 1.0 or casting variables:

In [ ]:
success_rate = (successful_orders * 1.0 / total_orders) * 100

Bug 2: Missing ax Parameter in Multi-Panel Seaborn Plots
The Problem:
When building subplots (plt.subplots(2, 2)), the AI forgot to pass the specific axis object into Seaborn:

In [ ]:
sns.countplot(data=df, x='category')  # Missing ax argument

This causes all graphs to overlap on the default canvas instead of organizing neatly into your grid cells.

The Human Fix:
Explicitly pass the target axis reference:

In [ ]:
sns.countplot(data=df, x='category', ax=axes[0, 0])

Question: What validation steps should a Data Analyst take before trusting or deploying AI-generated code?

Answer:

Check Shapes & Counts: Always check DataFrame shapes (df.shape) before and after data cleaning to ensure filtering or deduplication hasn't accidentally wiped out data.

Audit Null Values: Never assume missing values were handled properly—run df.isnull().sum() explicitly after imputation.

Verify SQL Joins: When performing LEFT JOIN operations, check row counts against the base table grain to ensure unexpected duplicate records haven't inflated your results.

Sanity Check Metrics: Compare automated calculations against manual descriptive stats (df.describe()) to catch logic flaws or hallucinations early.